In [ ]:
import robotic as ry
import numpy as np
import random

print("robotic:", ry.__version__, ry.compiled())


C = ry.Config()
C.addFile(ry.raiPath('scenarios/pandaSingle.g'))

table = C.addFrame("table")
table.setShape(ry.ST.box, size=[1, 1, .04])
table.setPosition([0, 0, 0])
table.setColor([.6, .6, .6])



def quat_from_rotmat(R):
    R = np.array(R, dtype=float)
    tr = np.trace(R)

    if tr > 0:
        S = np.sqrt(tr + 1.0) * 2.0
        w = 0.25 * S
        x = (R[2, 1] - R[1, 2]) / S
        y = (R[0, 2] - R[2, 0]) / S
        z = (R[1, 0] - R[0, 1]) / S
    elif (R[0, 0] > R[1, 1]) and (R[0, 0] > R[2, 2]):
        S = np.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2]) * 2.0
        w = (R[2, 1] - R[1, 2]) / S
        x = 0.25 * S
        y = (R[0, 1] + R[1, 0]) / S
        z = (R[0, 2] + R[2, 0]) / S
    elif R[1, 1] > R[2, 2]:
        S = np.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2]) * 2.0
        w = (R[0, 2] - R[2, 0]) / S
        x = (R[0, 1] + R[1, 0]) / S
        y = 0.25 * S
        z = (R[1, 2] + R[2, 1]) / S
    else:
        S = np.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1]) * 2.0
        w = (R[1, 0] - R[0, 1]) / S
        x = (R[0, 2] + R[2, 0]) / S
        y = (R[1, 2] + R[2, 1]) / S
        z = 0.25 * S

    q = np.array([w, x, y, z], dtype=float)
    q /= np.linalg.norm(q)
    return q


def normalize(v):
    v = np.array(v, dtype=float)
    n = np.linalg.norm(v)
    if n < 1e-12:
        return v
    return v / n


def make_rotation(closing_dir, approach_dir):
  
    x = normalize(closing_dir)
    z = normalize(approach_dir)
    y = normalize(np.cross(z, x))
    x = normalize(np.cross(y, z))
    R = np.column_stack([x, y, z])
    return R


def choose_approach_dir(axis):
   
    axis = normalize(axis)

    if abs(axis[0]) == 1:
        return np.array([0., 1., 0.])
    elif abs(axis[1]) == 1:
        return np.array([1., 0., 0.])
    else:
        return np.array([1., 0., 0.])


def generate_voxel_object(C, n_voxels=10, size=.08, seed=0):
    random.seed(seed)
    np.random.seed(seed)

    grid = {(0, 0, 0)}
    neighbors = [
        (1, 0, 0), (-1, 0, 0),
        (0, 1, 0), (0, -1, 0),
        (0, 0, 1), (0, 0, -1)
    ]

    while len(grid) < n_voxels:
        p = random.choice(list(grid))
        step = random.choice(neighbors)
        new = (p[0] + step[0], p[1] + step[1], p[2] + step[2])
        grid.add(new)

    root = C.addFrame("voxelObj")
    root.setPosition([0, 0, .12])

    frames = []
    voxel_map = {}

    for i, g in enumerate(grid):
        f = C.addFrame(f"cube{i}", "voxelObj")
        f.setShape(ry.ST.ssBox, size=[size, size, size, .001])
        f.setRelativePosition((np.array(g) * size).tolist())
        f.setColor(np.random.rand(3))
        frames.append(f)
        voxel_map[g] = f

    return frames, voxel_map, size

def lift_object_above_table(voxel_frames, table_z=0.0, voxel_size=0.08):
    min_z = 1e9
    for f in voxel_frames:
        z = f.getPosition()[2] - voxel_size/2
        min_z = min(min_z, z)

    if min_z < table_z:
        shift = table_z - min_z + 0.002
        for f in voxel_frames:
            p = f.getPosition()
            f.setPosition([p[0], p[1], p[2] + shift])

voxel_frames, voxel_map, voxel_size = generate_voxel_object(
    C, n_voxels=12, size=.08, seed=26
)

lift_object_above_table(voxel_frames)

def extract_open_faces(voxel_map, voxel_size):
   
    open_faces = []

    dirs = {
        ( 1, 0, 0): np.array([ 1., 0., 0.]),
        (-1, 0, 0): np.array([-1., 0., 0.]),
        ( 0, 1, 0): np.array([ 0., 1., 0.]),
        ( 0,-1, 0): np.array([ 0.,-1., 0.]),
        ( 0, 0, 1): np.array([ 0., 0., 1.]),
        ( 0, 0,-1): np.array([ 0., 0.,-1.]),
    }

    half = voxel_size / 2.0

    for g, f in voxel_map.items():
        center = np.array(f.getPosition(), dtype=float)

        for step, n in dirs.items():
            ngh = (g[0] + step[0], g[1] + step[1], g[2] + step[2])

            if ngh not in voxel_map:
                face_center = center + n * half
                open_faces.append({
                    "voxel": g,
                    "normal": n,
                    "face_center_world": face_center,
                    "grid_center": np.array(g, dtype=float)
                })

    return open_faces


open_faces = extract_open_faces(voxel_map, voxel_size)



def generate_surface_pair_grasps(
    C,
    open_faces,
    max_gripper_width=0.35,
    min_width=0.03,
    clearance=0.30,
    alignment_tol=0.03
):
    
    candidates = []
    candidate_id = 0

    axis_pairs = [
        (np.array([ 1., 0., 0.]), np.array([-1., 0., 0.])),
        (np.array([ 0., 1., 0.]), np.array([ 0.,-1., 0.])),
        (np.array([ 0., 0., 1.]), np.array([ 0., 0.,-1.])),
    ]

    for n_pos, n_neg in axis_pairs:
        pos_faces = [f for f in open_faces if np.allclose(f["normal"], n_pos)]
        neg_faces = [f for f in open_faces if np.allclose(f["normal"], n_neg)]

        axis = n_pos  # grasp closing direction

        for fa in pos_faces:
            for fb in neg_faces:
                pa = fa["face_center_world"]
                pb = fb["face_center_world"]

                # karşılıklı hizalama kontrolü
                if abs(axis[0]) == 1:
                    aligned = (abs(pa[1] - pb[1]) < alignment_tol) and (abs(pa[2] - pb[2]) < alignment_tol)
                elif abs(axis[1]) == 1:
                    aligned = (abs(pa[0] - pb[0]) < alignment_tol) and (abs(pa[2] - pb[2]) < alignment_tol)
                else:
                    aligned = (abs(pa[0] - pb[0]) < alignment_tol) and (abs(pa[1] - pb[1]) < alignment_tol)

                if not aligned:
                    continue

                diff = pb - pa
                width = abs(np.dot(diff, axis))

                if width < min_width or width > max_gripper_width:
                    continue

                contact_mid = 0.5 * (pa + pb)

                approach_dir = choose_approach_dir(axis)
                closing_dir = axis.copy()

                
                grasp_pos = contact_mid + clearance * approach_dir

                R = make_rotation(closing_dir, approach_dir)
                q = quat_from_rotmat(R)

                name = f"graspSurf{candidate_id}"
                g = C.addFrame(name)
                g.setShape(ry.ST.marker, size=[.10])
                g.setColor([0.2, 0.2, 1.0])
                g.setPosition(grasp_pos.tolist())
                g.setQuaternion(q.tolist())

                candidates.append({
                    "id": candidate_id,
                    "name": name,
                    "frame": g,
                    "pos": grasp_pos,
                    "contact_a": pa,
                    "contact_b": pb,
                    "contact_mid": contact_mid,
                    "approach_dir": approach_dir,
                    "closing_dir": closing_dir,
                    "required_width": width,
                    "score_geom": width,
                    "face_a": fa,
                    "face_b": fb,
                })

                candidate_id += 1

    return candidates


surface_grasps = generate_surface_pair_grasps(
    C,
    open_faces,
    max_gripper_width=0.35,
    min_width=0.03,
    clearance=0.30,
    alignment_tol=0.03
)

print("Total grasp candidates:", len(surface_grasps))



# scoring


def score_surface_grasps(candidates, voxel_frames):
    
    centers = np.array([np.array(f.getPosition(), dtype=float) for f in voxel_frames])
    com = centers.mean(axis=0)

    ranked = []
    for g in candidates:
        width = g["required_width"]
        dcom = np.linalg.norm(g["contact_mid"] - com)

        da = np.linalg.norm(g["contact_a"] - com)
        db = np.linalg.norm(g["contact_b"] - com)
        symmetry_penalty = abs(da - db)

        score = 0.0
        score -= 1.5 * width
        score -= 1.0 * dcom
        score -= 0.5 * symmetry_penalty

        gg = dict(g)
        gg["score"] = score
        ranked.append(gg)

    ranked.sort(key=lambda x: x["score"], reverse=True)
    return ranked


ranked_surface_grasps = score_surface_grasps(surface_grasps, voxel_frames)



# IK feasibility


def check_grasp_IK(C, grasp_frame_name):
    komo = ry.KOMO(C, 1, 1, 0, False)

    komo.addObjective(
        [], ry.FS.positionDiff,
        ['l_gripper', grasp_frame_name],
        ry.OT.eq, [1e1]
    )

    komo.addObjective(
        [], ry.FS.scalarProductZZ,
        ['l_gripper', grasp_frame_name],
        ry.OT.eq, [1e1], [1.]
    )

    komo.addObjective(
        [], ry.FS.accumulatedCollisions,
        [], ry.OT.eq
    )

    ret = ry.NLP_Solver(komo.nlp()).solve()
    return ret.feasible



# feasible filtering


feasible_surface_grasps = []

for g in ranked_surface_grasps:
    ok = check_grasp_IK(C, g["name"])
    print(
        g["name"],
        "width=", round(g["required_width"], 3),
        "score=", round(g["score"], 3),
        "->", ok
    )
    if ok:
        feasible_surface_grasps.append(g)

print("\nNum feasible surface grasps:", len(feasible_surface_grasps))


# select and visualize best grasp


if feasible_surface_grasps:
    best_grasp = feasible_surface_grasps[0]
    print("Best grasp selected:", best_grasp["name"])
    print("Best grasp width   :", round(best_grasp["required_width"], 3))
    print("Best grasp score   :", round(best_grasp["score"], 3))

    
    best_frame = C.getFrame(best_grasp["name"])
    best_frame.setColor([1, 1, 0])                # yellow
    best_frame.setShape(ry.ST.marker, size=[.25]) # larger

    
    best_ca = C.addFrame("best_contactA")
    best_ca.setShape(ry.ST.sphere, size=[.015])
    best_ca.setColor([0, 0, 0])
    best_ca.setPosition(best_grasp["contact_a"].tolist())

    best_cb = C.addFrame("best_contactB")
    best_cb.setShape(ry.ST.sphere, size=[.015])
    best_cb.setColor([0, 0, 0])
    best_cb.setPosition(best_grasp["contact_b"].tolist())


    print("\nBest grasp contact points:")

    print("Contact A:", np.round(best_grasp["contact_a"], 4))
    print("Contact B:", np.round(best_grasp["contact_b"], 4))

    print("Contact midpoint:", np.round(best_grasp["contact_mid"], 4))

    print("Approach direction:", np.round(best_grasp["approach_dir"], 4))
    print("Closing direction :", np.round(best_grasp["closing_dir"], 4))

else:
    best_grasp = None
    print("No feasible grasp found.")

C.view()

robotic: 0.2.2 compile time: Nov 26 2024 16:50:51
-- WARNING:kin.cpp:addFrame:195(-1) frame already exists! returning existing without modifications!
Total grasp candidates: 22
graspSurf10 width= 0.08 score= -0.185 -> True
graspSurf0 width= 0.08 score= -0.206 -> True
graspSurf12 width= 0.08 score= -0.218 -> True
graspSurf5 width= 0.08 score= -0.223 -> True
graspSurf8 width= 0.08 score= -0.224 -> True
graspSurf1 width= 0.08 score= -0.228 -> True
graspSurf9 width= 0.08 score= -0.242 -> True
graspSurf11 width= 0.08 score= -0.263 -> True
graspSurf2 width= 0.08 score= -0.263 -> False
graspSurf20 width= 0.08 score= -0.264 -> True
graspSurf17 width= 0.08 score= -0.265 -> True
graspSurf15 width= 0.08 score= -0.296 -> True
graspSurf6 width= 0.08 score= -0.299 -> True
graspSurf14 width= 0.16 score= -0.313 -> True
graspSurf19 width= 0.16 score= -0.327 -> True
graspSurf16 width= 0.16 score= -0.34 -> True
graspSurf13 width= 0.16 score= -0.342 -> True
graspSurf18 width= 0.16 score= -0.36 -> True
gra

0

-- WARNING:RenderData.cpp:glInitialize:114(-1) FreeType Error: Failed to load font 'ubuntu/Ubuntu-L.ttf' error code: 1 -> text rendering disabled


In [6]:
for n in C.getFrameNames():
    if "finger" in n.lower():
        print(n)

l_panda_finger_joint1_origin
l_panda_finger_joint2_origin
l_panda_finger_joint1
l_panda_finger_joint2
l_panda_leftfinger
l_panda_rightfinger
l_panda_leftfinger_0
l_panda_rightfinger_0
l_finger1
l_finger2


In [ ]:
import robotic as ry
import numpy as np
import random

print("robotic:", ry.__version__, ry.compiled())

C = ry.Config()
C.addFile(ry.raiPath('scenarios/pandaSingle.g'))

table = C.addFrame("table")
table.setShape(ry.ST.box, size=[1,1,.04])
table.setPosition([0,0,0])
table.setColor([.6,.6,.6])



def quat_from_rotmat(R):
    R = np.array(R, dtype=float)
    tr = np.trace(R)

    if tr > 0:
        S = np.sqrt(tr + 1.0) * 2.0
        w = 0.25 * S
        x = (R[2, 1] - R[1, 2]) / S
        y = (R[0, 2] - R[2, 0]) / S
        z = (R[1, 0] - R[0, 1]) / S
    elif (R[0, 0] > R[1, 1]) and (R[0, 0] > R[2, 2]):
        S = np.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2]) * 2.0
        w = (R[2, 1] - R[1, 2]) / S
        x = 0.25 * S
        y = (R[0, 1] + R[1, 0]) / S
        z = (R[0, 2] + R[2, 0]) / S
    elif R[1, 1] > R[2, 2]:
        S = np.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2]) * 2.0
        w = (R[0, 2] - R[2, 0]) / S
        x = (R[0, 1] + R[1, 0]) / S
        y = 0.25 * S
        z = (R[1, 2] + R[2, 1]) / S
    else:
        S = np.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1]) * 2.0
        w = (R[1, 0] - R[0, 1]) / S
        x = (R[0, 2] + R[2, 0]) / S
        y = (R[1, 2] + R[2, 1]) / S
        z = 0.25 * S

    q = np.array([w, x, y, z], dtype=float)
    q /= np.linalg.norm(q)
    return q


def normalize(v):
    v = np.array(v, dtype=float)
    n = np.linalg.norm(v)
    if n < 1e-12:
        return v
    return v / n


def make_rotation(closing_dir, approach_dir):
    
    x = normalize(closing_dir)
    z = normalize(approach_dir)
    y = normalize(np.cross(z, x))
    x = normalize(np.cross(y, z))
    R = np.column_stack([x, y, z])
    return R


def set_frame_pose(frame, pos, quat):
    px, py, pz = pos.tolist()
    qw, qx, qy, qz = quat.tolist()
    pose_str = f"t({px} {py} {pz}) d({qw} {qx} {qy} {qz})"
    frame.setPose(pose_str)



def generate_voxel_object(C, n_voxels=10, size=.08, seed=0):
    random.seed(seed)
    np.random.seed(seed)

    grid = {(0,0,0)}
    neighbors = [(1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)]

    while len(grid) < n_voxels:
        p = random.choice(list(grid))
        step = random.choice(neighbors)
        new = (p[0]+step[0], p[1]+step[1], p[2]+step[2])
        grid.add(new)

    root = C.addFrame("voxelObj")
    root.setPosition([0,0,.12])

    frames = []
    voxel_map = {}

    for i, g in enumerate(grid):
        f = C.addFrame(f"cube{i}", "voxelObj")
        f.setShape(ry.ST.ssBox, size=[size,size,size,.001])
        f.setRelativePosition((np.array(g)*size).tolist())
        f.setColor(np.random.rand(3))
        frames.append(f)
        voxel_map[g] = f

    return frames, voxel_map, size


voxel_frames, voxel_map, voxel_size = generate_voxel_object(C, n_voxels=12, size=.08, seed=4)



def extract_open_faces(voxel_map, voxel_size):
    open_faces = []

    dirs = {
        ( 1, 0, 0): np.array([ 1., 0., 0.]),
        (-1, 0, 0): np.array([-1., 0., 0.]),
        ( 0, 1, 0): np.array([ 0., 1., 0.]),
        ( 0,-1, 0): np.array([ 0.,-1., 0.]),
        ( 0, 0, 1): np.array([ 0., 0., 1.]),
        ( 0, 0,-1): np.array([ 0., 0.,-1.]),
    }

    half = voxel_size / 2.0

    for g, f in voxel_map.items():
        center = np.array(f.getPosition(), dtype=float)

        for step, n in dirs.items():
            ngh = (g[0] + step[0], g[1] + step[1], g[2] + step[2])
            if ngh not in voxel_map:
                face_center = center + n * half
                open_faces.append({
                    "voxel": g,
                    "normal": n,
                    "face_center_world": face_center,
                })

    return open_faces


open_faces = extract_open_faces(voxel_map, voxel_size)



def choose_approach_dir(closing_dir):
    """
    closing_dir'a dik ve masaüstü için mantıklı bir yaklaşım yönü seç.
    """
    closing_dir = normalize(closing_dir)

    
    up = np.array([0., 0., 1.])

  
    if abs(np.dot(closing_dir, up)) > 0.9:
        side = np.array([1., 0., 0.])
        approach_dir = normalize(np.cross(closing_dir, side))
    else:
        approach_dir = normalize(np.cross(closing_dir, up))

    
    if approach_dir[2] < -0.2:
        approach_dir = -approach_dir

    return approach_dir


def generate_real_grasp_candidates(
    C,
    open_faces,
    min_width=0.03,
    max_width=0.35,
    alignment_tol=0.03,
    pregrasp_clearance=0.14,
):
    candidates = []
    cid = 0

    axis_pairs = [
        (np.array([ 1.,0.,0.]), np.array([-1.,0.,0.])),
        (np.array([ 0.,1.,0.]), np.array([ 0.,-1.,0.])),
        (np.array([ 0.,0.,1.]), np.array([ 0.,0.,-1.])),
    ]

    for n_pos, n_neg in axis_pairs:
        pos_faces = [f for f in open_faces if np.allclose(f["normal"], n_pos)]
        neg_faces = [f for f in open_faces if np.allclose(f["normal"], n_neg)]

        closing_axis = normalize(n_pos)

        for fa in pos_faces:
            for fb in neg_faces:
                pa = fa["face_center_world"]   # contact point A
                pb = fb["face_center_world"]   # contact point B

              
                if abs(closing_axis[0]) == 1:
                    aligned = abs(pa[1]-pb[1]) < alignment_tol and abs(pa[2]-pb[2]) < alignment_tol
                elif abs(closing_axis[1]) == 1:
                    aligned = abs(pa[0]-pb[0]) < alignment_tol and abs(pa[2]-pb[2]) < alignment_tol
                else:
                    aligned = abs(pa[0]-pb[0]) < alignment_tol and abs(pa[1]-pb[1]) < alignment_tol

                if not aligned:
                    continue

                width = abs(np.dot(pb - pa, closing_axis))
                if width < min_width or width > max_width:
                    continue

                grasp_center = 0.5 * (pa + pb)
                closing_dir = normalize(pb - pa)
                approach_dir = choose_approach_dir(closing_dir)

                # final grasp pose = contact center
                grasp_pos = grasp_center

                # pregrasp = objeden biraz uzakta
                pregrasp_pos = grasp_center + pregrasp_clearance * approach_dir

                R = make_rotation(closing_dir, approach_dir)
                q = quat_from_rotmat(R)

                grasp_name = f"graspReal{cid}"
                pre_name = f"pregraspReal{cid}"

                # final grasp frame
                gf = C.addFrame(grasp_name)
                gf.setShape(ry.ST.marker, size=[.12])
                gf.setColor([1, 0, 0])
                set_frame_pose(gf, grasp_pos, q)

                # pregrasp frame
                pf = C.addFrame(pre_name)
                pf.setShape(ry.ST.marker, size=[.10])
                pf.setColor([0, 0, 1])
                set_frame_pose(pf, pregrasp_pos, q)

                # contact markers
                cfa = C.addFrame(f"contactA_{cid}")
                cfa.setShape(ry.ST.marker, size=[.06])
                cfa.setColor([0, 1, 0])
                cfa.setPosition(pa.tolist())

                cfb = C.addFrame(f"contactB_{cid}")
                cfb.setShape(ry.ST.marker, size=[.06])
                cfb.setColor([0, 1, 0])
                cfb.setPosition(pb.tolist())

                candidates.append({
                    "id": cid,
                    "pregrasp_name": pre_name,
                    "grasp_name": grasp_name,
                    "contact_a": pa,
                    "contact_b": pb,
                    "grasp_center": grasp_center,
                    "closing_dir": closing_dir,
                    "approach_dir": approach_dir,
                    "required_width": width,
                    "score_geom": 0.0,
                })
                cid += 1

    return candidates


real_grasps = generate_real_grasp_candidates(
    C,
    open_faces,
    min_width=0.03,
    max_width=0.35,
    alignment_tol=0.03,
    pregrasp_clearance=0.14,
)


def score_real_grasps(candidates, voxel_frames):
    centers = np.array([np.array(f.getPosition(), dtype=float) for f in voxel_frames])
    com = centers.mean(axis=0)

    ranked = []
    for g in candidates:
        width = g["required_width"]
        dcom = np.linalg.norm(g["grasp_center"] - com)

        score = 0.0
        score -= 1.5 * width
        score -= 1.0 * dcom

        
        score += 0.3 * max(g["approach_dir"][2], 0.0)

        gg = dict(g)
        gg["score"] = score
        ranked.append(gg)

    ranked.sort(key=lambda x: x["score"], reverse=True)
    return ranked


ranked_real_grasps = score_real_grasps(real_grasps, voxel_frames)



def ik_pose_feasible(C, target_frame_name):
    komo = ry.KOMO(C, 1, 1, 0, False)

    komo.addObjective(
        [], ry.FS.positionDiff,
        ['l_gripper', target_frame_name],
        ry.OT.eq, [1e1]
    )

    komo.addObjective(
        [], ry.FS.scalarProductZZ,
        ['l_gripper', target_frame_name],
        ry.OT.eq, [1e1], [1.]
    )

    komo.addObjective(
        [], ry.FS.accumulatedCollisions,
        [], ry.OT.eq
    )

    ret = ry.NLP_Solver(komo.nlp()).solve()
    return ret.feasible


feasible_real_grasps = []
for g in ranked_real_grasps:
    ok_pre = ik_pose_feasible(C, g["pregrasp_name"])
    ok_grasp = ik_pose_feasible(C, g["grasp_name"])
    ok = ok_pre and ok_grasp

    print(
        g["grasp_name"],
        "width=", round(g["required_width"], 3),
        "score=", round(g["score"], 3),
        "pre=", ok_pre,
        "grasp=", ok_grasp,
        "->", ok
    )

    if ok:
        feasible_real_grasps.append(g)

print("\nNum feasible real grasps:", len(feasible_real_grasps))

if feasible_real_grasps:
    best = feasible_real_grasps[0]
    C.getFrame(best["grasp_name"]).setColor([1, 1, 0])      
    C.getFrame(best["pregrasp_name"]).setColor([0, 1, 1])   
    print("Best grasp:", best["grasp_name"])

C.view()

robotic: 0.2.2 compile time: Nov 26 2024 16:50:51
-- WARNING:kin.cpp:addFrame:195(-1) frame already exists! returning existing without modifications!
graspReal18 width= 0.08 score= -0.181 pre= False grasp= False -> False
graspReal4 width= 0.08 score= -0.209 pre= True grasp= True -> True
graspReal12 width= 0.08 score= -0.209 pre= True grasp= True -> True
graspReal16 width= 0.08 score= -0.209 pre= False grasp= True -> False
graspReal19 width= 0.08 score= -0.22 pre= False grasp= False -> False
graspReal1 width= 0.08 score= -0.226 pre= True grasp= False -> False
graspReal8 width= 0.08 score= -0.226 pre= False grasp= False -> False
graspReal21 width= 0.08 score= -0.244 pre= False grasp= False -> False
graspReal5 width= 0.08 score= -0.256 pre= True grasp= True -> True
graspReal13 width= 0.08 score= -0.256 pre= True grasp= True -> True
graspReal11 width= 0.08 score= -0.26 pre= False grasp= True -> False
graspReal20 width= 0.08 score= -0.26 pre= False grasp= True -> False
graspReal6 width= 0.0

0

-- WARNING:RenderData.cpp:glInitialize:114(-1) FreeType Error: Failed to load font 'ubuntu/Ubuntu-L.ttf' error code: 1 -> text rendering disabled
